# UC4: AI-Powered Executive Insights
### Prime Insurance — Gold Layer Business Intelligence

This notebook:
- Reads from all 5 Gold layer tables
- Computes business KPIs via aggregation (never passes raw rows to LLM)
- Generates AI-written executive summaries for 4 business domains
- Demonstrates `ai_query()` with `databricks-gpt-oss-20b` and `get_json_object()` extraction
- Writes output to `primeins.gold.ai_business_insights`

## 0. Setup & Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import json
from datetime import datetime

# ── Change only this line to point at a different catalog ──────────────────────
CATALOG = "primeins"
# ──────────────────────────────────────────────────────────────────────────────

GOLD         = f"{CATALOG}.gold"
OUTPUT_TABLE = "primeins.gold.ai_business_insights"
AI_MODEL     = "databricks-gpt-oss-20b"

print(f"Catalog  : {CATALOG}")
print(f"AI Model : {AI_MODEL}")
print(f"Output   : {OUTPUT_TABLE}")

## 1. Load Gold Layer Tables (current records only)

In [0]:
dim_policy   = spark.read.table(f"{GOLD}.dim_policy").filter("is_current = true")
dim_customer = spark.read.table(f"{GOLD}.dim_customer").filter("is_current = true")
dim_car      = spark.read.table(f"{GOLD}.dim_car").filter("is_current = true")
fact_claims  = spark.read.table(f"{GOLD}.fact_claims")
fact_sales   = spark.read.table(f"{GOLD}.fact_car_sales")

print("All 5 Gold tables loaded.")

---
## 2. KPI Aggregation
> **Rule**: Aggregate first — COUNT, AVG, SUM, distributions. Never pass raw rows to the LLM.

### 2a. Domain 1 — Policy Portfolio KPIs

In [0]:
policy_kpis = {}

overview = dim_policy.agg(
    F.count("*").alias("total_policies"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.round(F.avg("policy_annual_premium"), 2).alias("avg_annual_premium"),
    F.round(F.sum("policy_annual_premium"), 2).alias("total_annual_premium"),
    F.round(F.avg("policy_deductable"), 2).alias("avg_deductable"),
    F.round(F.avg("umbrella_limit"), 2).alias("avg_umbrella_limit"),
    F.sum(F.col("_umbrella_limit_zero_flag").cast("int")).alias("zero_umbrella_count"),
).collect()[0]
policy_kpis["overview"] = overview.asDict()

state_dist = (
    dim_policy.groupBy("policy_state_full")
    .agg(
        F.count("*").alias("policy_count"),
        F.round(F.avg("policy_annual_premium"), 2).alias("avg_premium"),
    )
    .orderBy(F.desc("policy_count"))
    .limit(5)
    .collect()
)
policy_kpis["top5_states"] = [r.asDict() for r in state_dist]

csl_dist = (
    dim_policy.groupBy("policy_csl")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .collect()
)
policy_kpis["csl_distribution"] = [r.asDict() for r in csl_dist]

premium_buckets = (
    dim_policy
    .withColumn("premium_bucket",
        F.when(F.col("policy_annual_premium") < 500,  "<$500")
         .when(F.col("policy_annual_premium") < 1000, "$500-$1k")
         .when(F.col("policy_annual_premium") < 2000, "$1k-$2k")
         .otherwise(">$2k"))
    .groupBy("premium_bucket")
    .agg(F.count("*").alias("count"))
    .orderBy("premium_bucket")
    .collect()
)
policy_kpis["premium_buckets"] = [r.asDict() for r in premium_buckets]

print("Policy KPIs computed:")
print(json.dumps(policy_kpis, indent=2, default=str))

### 2b. Domain 2 — Claims Performance KPIs

In [0]:
claims_kpis = {}

claims_overview = fact_claims.agg(
    F.count("*").alias("total_claims"),
    F.sum("is_rejected_int").alias("total_rejected"),
    F.round(F.avg("is_rejected_int") * 100, 2).alias("rejection_rate_pct"),
    F.round(F.avg("processing_time_days"), 2).alias("avg_processing_days"),
    F.max("processing_time_days").alias("max_processing_days"),
    F.min("processing_time_days").alias("min_processing_days"),
).collect()[0]
claims_kpis["overview"] = claims_overview.asDict()

severity_dist = (
    fact_claims.groupBy("incident_severity")
    .agg(
        F.count("*").alias("claim_count"),
        F.round(F.avg("processing_time_days"), 2).alias("avg_processing_days"),
        F.sum("is_rejected_int").alias("rejected_count"),
        F.round(F.avg("is_rejected_int") * 100, 2).alias("rejection_rate_pct"),
    )
    .orderBy(F.desc("claim_count"))
    .collect()
)
claims_kpis["by_severity"] = [r.asDict() for r in severity_dist]

state_claims = (
    fact_claims.groupBy("incident_state_full")
    .agg(
        F.count("*").alias("claim_count"),
        F.round(F.avg("processing_time_days"), 2).alias("avg_processing_days"),
        F.round(F.avg("is_rejected_int") * 100, 2).alias("rejection_rate_pct"),
    )
    .orderBy(F.desc("claim_count"))
    .limit(5)
    .collect()
)
claims_kpis["top5_states"] = [r.asDict() for r in state_claims]

multi_vehicle = (
    fact_claims.groupBy("number_of_vehicles_involved")
    .agg(F.count("*").alias("count"))
    .orderBy("number_of_vehicles_involved")
    .collect()
)
claims_kpis["vehicles_involved_distribution"] = [r.asDict() for r in multi_vehicle]

print("Claims KPIs computed:")
print(json.dumps(claims_kpis, indent=2, default=str))

### 2c. Domain 3 — Customer Profile KPIs

In [0]:
customer_kpis = {}

cust_overview = dim_customer.agg(
    F.count("*").alias("total_customers"),
    F.round(F.avg("balance"), 2).alias("avg_balance"),
    F.round(F.sum("balance"), 2).alias("total_balance"),
    F.round(F.max("balance"), 2).alias("max_balance"),
    F.round(F.min("balance"), 2).alias("min_balance"),
).collect()[0]
customer_kpis["overview"] = cust_overview.asDict()

job_dist = (
    dim_customer.groupBy("job")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
    )
    .orderBy(F.desc("count"))
    .limit(6)
    .collect()
)
customer_kpis["top6_jobs"] = [r.asDict() for r in job_dist]

marital_dist = (
    dim_customer.groupBy("marital")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
    )
    .orderBy(F.desc("count"))
    .collect()
)
customer_kpis["marital_distribution"] = [r.asDict() for r in marital_dist]

edu_dist = (
    dim_customer.groupBy("education")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .collect()
)
customer_kpis["education_distribution"] = [r.asDict() for r in edu_dist]

product_cross = (
    dim_customer.groupBy("carloan", "hhinsurance")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .collect()
)
customer_kpis["product_cross_sell"] = [r.asDict() for r in product_cross]

region_dist = (
    dim_customer.groupBy("region")
    .agg(
        F.count("*").alias("count"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
    )
    .orderBy(F.desc("count"))
    .collect()
)
customer_kpis["region_distribution"] = [r.asDict() for r in region_dist]

print("Customer KPIs computed:")
print(json.dumps(customer_kpis, indent=2, default=str))

### 2d. Domain 4 — Vehicle Fleet & Sales KPIs

In [0]:
vehicle_kpis = {}

car_overview = dim_car.agg(
    F.count("*").alias("total_vehicles"),
    F.round(F.avg("km_driven"), 0).alias("avg_km_driven"),
    F.round(F.avg("mileage"), 2).alias("avg_mileage"),
    F.round(F.avg("engine_cc"), 0).alias("avg_engine_cc"),
    F.round(F.avg("max_power_bhp"), 2).alias("avg_max_power_bhp"),
).collect()[0]
vehicle_kpis["car_overview"] = car_overview.asDict()

fuel_dist = (
    dim_car.groupBy("fuel")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .collect()
)
vehicle_kpis["fuel_distribution"] = [r.asDict() for r in fuel_dist]

trans_dist = (
    dim_car.groupBy("transmission")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .collect()
)
vehicle_kpis["transmission_distribution"] = [r.asDict() for r in trans_dist]

sales_overview = fact_sales.agg(
    F.count("*").alias("total_listings"),
    F.sum("is_unsold").alias("unsold_count"),
    F.round((1 - F.avg("is_unsold")) * 100, 2).alias("sold_rate_pct"),
    F.round(F.avg("original_selling_price"), 2).alias("avg_selling_price"),
    F.round(F.sum("original_selling_price"), 2).alias("total_gmv"),
    F.round(F.avg("days_on_market"), 1).alias("avg_days_on_market"),
).collect()[0]
vehicle_kpis["sales_overview"] = sales_overview.asDict()

sales_region = (
    fact_sales.groupBy("region")
    .agg(
        F.count("*").alias("listings"),
        F.round(F.avg("original_selling_price"), 2).alias("avg_price"),
        F.round(F.avg("days_on_market"), 1).alias("avg_days_on_market"),
        F.round((1 - F.avg("is_unsold")) * 100, 2).alias("sold_rate_pct"),
    )
    .orderBy(F.desc("listings"))
    .collect()
)
vehicle_kpis["sales_by_region"] = [r.asDict() for r in sales_region]

print("Vehicle & Sales KPIs computed:")
print(json.dumps(vehicle_kpis, indent=2, default=str))

---
## 3. AI Executive Summaries

`ai_query('databricks-gpt-oss-20b', prompt)` returns plain text directly.
`_extract_text()` handles both plain-text and JSON-array responses safely.

In [0]:
def _extract_text(raw) -> str:
    """
    Safely extracts plain text from an ai_query() response.
    Case 1 — ai_query returns plain text  ->  return as-is.
    Case 2 — ai_query returns a JSON array -> parse and pull $.text from $[1].
    """
    if raw is None:
        return ""
    text = str(raw).strip()
    if text.startswith("["):
        try:
            parsed = json.loads(text)
            return parsed[1].get("text", text)
        except Exception:
            return text
    return text


def call_llm(domain: str, kpis: dict, instructions: str) -> str:
    """
    Calls ai_query('databricks-gpt-oss-20b', prompt) via spark.sql().
    Aggregated KPI JSON is embedded in the prompt — no raw rows ever passed.
    Response is cleaned via _extract_text().
    """
    kpi_str    = json.dumps(kpis, default=str).replace("'", '"')
    prompt     = (
        f"You are a senior insurance analyst writing an executive summary for C-suite leaders. "
        f"Be direct, concise, and data-driven. Write 3-5 short paragraphs. "
        f"No disclaimers or markdown headers. "
        f"Domain: {domain}. "
        f"Aggregated KPI data: {kpi_str}. "
        f"{instructions}"
    )
    prompt_sql = prompt.replace("'", "\\'")
    result_row = spark.sql(
        f"SELECT ai_query('{AI_MODEL}', '{prompt_sql}') AS response"
    ).collect()[0]
    return _extract_text(result_row["response"])


print("_extract_text() and call_llm() defined.")

### 3a. Policy Portfolio Executive Summary

In [0]:
policy_summary = call_llm(
    domain="Policy Portfolio",
    kpis=policy_kpis,
    instructions=(
        "Summarise: portfolio size and total premium revenue; "
        "top state concentrations and geographic risk exposure; "
        "coverage adequacy across deductibles, umbrella limits, and CSL mix; "
        "key risks and growth opportunities for leadership to act on."
    ),
)
print("=== POLICY PORTFOLIO EXECUTIVE SUMMARY ===")
print(policy_summary)

### 3b. Claims Performance Executive Summary

In [0]:
claims_summary = call_llm(
    domain="Claims Performance",
    kpis=claims_kpis,
    instructions=(
        "Summarise: total claim volume and rejection rate and what they signal operationally; "
        "processing time efficiency and where bottlenecks may exist; "
        "geographic and severity-level patterns that require leadership attention; "
        "specific operational improvements to prioritise."
    ),
)
print("=== CLAIMS PERFORMANCE EXECUTIVE SUMMARY ===")
print(claims_summary)

### 3c. Customer Profile Executive Summary

In [0]:
customer_summary = call_llm(
    domain="Customer Profile",
    kpis=customer_kpis,
    instructions=(
        "Summarise: customer demographics and financial profile; "
        "cross-sell penetration rates for car loans and household insurance and the opportunity gap; "
        "regional concentration risks and untapped markets; "
        "retention risks and acquisition opportunities."
    ),
)
print("=== CUSTOMER PROFILE EXECUTIVE SUMMARY ===")
print(customer_summary)

### 3d. Vehicle Fleet & Sales Executive Summary

In [0]:
vehicle_summary = call_llm(
    domain="Vehicle Fleet and Sales",
    kpis=vehicle_kpis,
    instructions=(
        "Summarise: fleet composition by fuel type and transmission and underwriting implications; "
        "sales velocity including average days on market and sold rate; "
        "regional market performance differences; "
        "product and pricing recommendations for the insurance portfolio."
    ),
)
print("=== VEHICLE FLEET & SALES EXECUTIVE SUMMARY ===")
print(vehicle_summary)

--
## 4. `ai_query()` Demonstrations — SQL Native

Syntax: `ai_query('databricks-gpt-oss-20b', prompt)`

> **Note on response format:**
> When using `ai_query()` with an OpenAI-compatible external model endpoint,
> the response is a JSON array and we need to extract the text field manually:

> However, with `databricks-gpt-oss-20b` the model returns **plain text directly**,
> so no JSON extraction is needed — `ai_query(...)` is used as-is.


### Example 1 — Per-Fuel-Type Underwriting Risk Assessment

In [0]:
%sql
WITH fuel_risk AS (
  SELECT
    c.fuel,
    COUNT(*)                                      AS total_vehicles,
    ROUND(AVG(c.km_driven), 0)                    AS avg_km_driven,
    ROUND(AVG(c.max_power_bhp), 1)                AS avg_power_bhp,
    COUNT(cl.claim_id)                            AS total_claims
  FROM
    primeins.gold.dim_car     c
    LEFT JOIN primeins.gold.fact_claims cl
           ON c.car_sk = cl.policy_sk
  WHERE
    c.is_current = TRUE
  GROUP BY
    c.fuel
)
SELECT
  fuel,
  total_vehicles,
  avg_km_driven,
  avg_power_bhp,
  total_claims,
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'In 1-2 sentences, assess underwriting risk for ', fuel, ' vehicles. ',
      'Stats: total_vehicles=', CAST(total_vehicles AS STRING),
      ', avg_km_driven=',       CAST(avg_km_driven  AS STRING),
      ', avg_power_bhp=',       CAST(avg_power_bhp  AS STRING),
      ', total_claims=',        CAST(total_claims   AS STRING),
      '. Be direct. No disclaimers.'
    )
  ) AS ai_underwriting_insight
FROM
  fuel_risk
ORDER BY
  total_claims DESC;

### Example 2 — Per-Segment Customer Retention Tactic

In [0]:
%sql
WITH customer_segments AS (
  SELECT
    job,
    COUNT(*)                                                                           AS customer_count,
    ROUND(AVG(balance), 0)                                                             AS avg_balance,
    ROUND(SUM(CASE WHEN carloan     = 'yes' THEN 1 ELSE 0 END) / COUNT(*) * 100, 1)   AS carloan_pct,
    ROUND(SUM(CASE WHEN hhinsurance = 'yes' THEN 1 ELSE 0 END) / COUNT(*) * 100, 1)   AS hhinsurance_pct
  FROM
    primeins.gold.dim_customer
  WHERE
      is_current = TRUE
    AND job IS NOT NULL
  GROUP BY
    job
  ORDER BY
    customer_count DESC
  LIMIT 6
)
SELECT
  job,
  customer_count,
  avg_balance,
  carloan_pct,
  hhinsurance_pct,
  ai_query(
    'databricks-gpt-oss-20b',
    CONCAT(
      'Write one specific actionable retention or upsell tactic for an insurance company ',
      'targeting the "', job, '" customer segment. ',
      'Context: avg_balance=$',                CAST(avg_balance     AS STRING),
      ', car_loan_penetration=',               CAST(carloan_pct     AS STRING), '%',
      ', household_insurance_penetration=',    CAST(hhinsurance_pct AS STRING), '%.',
      ' One sentence only. No disclaimers.'
    )
  ) AS ai_retention_tactic
FROM
  customer_segments
ORDER BY
  customer_count DESC;

---
## 5. Create Output Table & Write Results

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS primeins.gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS primeins.gold.ai_business_insights (
  insight_id        STRING    COMMENT 'Unique row identifier',
  domain            STRING    COMMENT 'policy_portfolio | claims_performance | customer_profile | vehicle_fleet_sales',
  summary_title     STRING    COMMENT 'Short descriptive title',
  executive_summary STRING    COMMENT 'AI-generated executive summary',
  kpi_json          STRING    COMMENT 'Aggregated KPIs passed to the LLM - for auditability',
  generation_ts     TIMESTAMP COMMENT 'UTC timestamp when this row was generated',
  model_name        STRING    COMMENT 'Foundation model used',
  status            STRING    COMMENT 'SUCCESS | ERROR'
)
USING DELTA
COMMENT 'AI-generated executive business insights from Gold layer KPI aggregations';

In [0]:
insight_schema = StructType([
    StructField("insight_id",        StringType(),    False),
    StructField("domain",            StringType(),    True),
    StructField("summary_title",     StringType(),    True),
    StructField("executive_summary", StringType(),    True),
    StructField("kpi_json",          StringType(),    True),
    StructField("generation_ts",     TimestampType(), True),
    StructField("model_name",        StringType(),    True),
    StructField("status",            StringType(),    True),
])

now = datetime.utcnow()

def make_row(insight_id, domain, title, summary, kpis):
    status = "SUCCESS" if summary and not str(summary).startswith("ERROR") else "ERROR"
    return (insight_id, domain, title, summary, json.dumps(kpis, default=str), now, AI_MODEL, status)

rows = [
    make_row("INS-001", "policy_portfolio",    "Policy Portfolio Health & Geographic Exposure",             policy_summary,   policy_kpis),
    make_row("INS-002", "claims_performance",  "Claims Operations: Throughput, Rejection & Risk Hotspots",  claims_summary,   claims_kpis),
    make_row("INS-003", "customer_profile",    "Customer Demographic & Cross-Sell Opportunity Analysis",    customer_summary, customer_kpis),
    make_row("INS-004", "vehicle_fleet_sales", "Vehicle Fleet Composition & Sales Market Dynamics",         vehicle_summary,  vehicle_kpis),
]

df_insights = spark.createDataFrame(rows, schema=insight_schema)
# df_insights.write.mode("overwrite").saveAsTable('primeins.gold.ai_business_insights')
display(df_insights)

## 6. Verify Output

In [0]:
%sql
SELECT
  insight_id,
  domain,
  summary_title,
  executive_summary,
  generation_ts,
  model_name,
  status
FROM
  primeins.gold.ai_business_insights
ORDER BY
  insight_id;

---
## Summary

| Step | What was done |
|------|--------------|
| **Catalog variable** | `CATALOG = "primeins"` — set once, used everywhere |
| **KPI Aggregation** | COUNT / AVG / SUM / distributions across all 5 Gold tables — zero raw rows sent to LLM |
| **`_extract_text()`** | Handles both plain-text and JSON-array responses from `ai_query()` |
| **Domain 1 — Policy Portfolio** | Premium revenue, state distribution, CSL & deductible analysis |
| **Domain 2 — Claims Performance** | Rejection rates, processing times, severity & state breakdowns |
| **Domain 3 — Customer Profile** | Demographics, balance, cross-sell penetration, regional spread |
| **Domain 4 — Vehicle Fleet & Sales** | Fleet composition, fuel/transmission mix, sales velocity |
| **`ai_query()` Example 1** | Per-state risk narrative — `get_json_object(get_json_object(ai_query(...), '$[1]'), '$.text')` |
| **`ai_query()` Example 2** | Per-job-segment retention tactic — same extraction pattern |
| **HTML Dashboard** | `displayHTML()` renders all 4 summaries as a styled executive briefing |
| **Output Table** | `primeins.gold.ai_business_insights` — 4 rows, Delta, `kpi_json` for full auditability |